# Save/Load Advanced

Audit coverage for unified .tlspec manifests, polymorphic loading, and validation.

In [ ]:
# COVERS: torchlens.load, torchlens.ModelLog.save, torchlens.Bundle.save
# COVERS: torchlens.io.inspect_tlspec, torchlens.validation.validate_tlspec
from pathlib import Path
import tempfile

import torch
from torch import nn

import torchlens as tl
from torchlens.options import CaptureOptions, VisualizationOptions
from torchlens.validation import validate_tlspec


class SaveLoadAuditModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)

    def forward(self, x):
        return torch.relu(self.linear(x))


torch.manual_seed(20)
log = tl.log_forward_pass(
    SaveLoadAuditModel().eval(),
    torch.randn(1, 2),
    capture=CaptureOptions(intervention_ready=True, layers_to_save="all", random_seed=0),
    visualization=VisualizationOptions(view="none"),
)

with tempfile.TemporaryDirectory() as tmp_dir:
    root = Path(tmp_dir)
    model_path = root / "model.tlspec"
    bundle_path = root / "bundle.tlspec"
    intervention_path = root / "intervention.tlspec"

    log.save(model_path, level="portable")
    bundle = tl.bundle({"baseline": log}, baseline="baseline")
    bundle.save(bundle_path, level="portable")
    log.set(tl.func("relu"), tl.zero_ablate(), confirm_mutation=True)
    log.save_intervention(intervention_path, level="portable")

    for path, kind in [
        (model_path, "model_log"),
        (bundle_path, "bundle"),
        (intervention_path, "intervention"),
    ]:
        manifest = tl.io.inspect_tlspec(path)
        assert manifest["kind"] == kind
        assert manifest["tlspec_version"] == 1
        validate_tlspec(path)
        assert tl.io.detect_tlspec_format(path) == "v2.0_unified"

    assert isinstance(tl.load(model_path), tl.ModelLog)
    assert isinstance(tl.load(bundle_path), tl.Bundle)
    assert tl.load(intervention_path).metadata["save_level"] == "portable"